## Exercise 5: Geospatial wrangling and making maps

Skills: 
* More geospatial practice building on earlier skills
* Make a map with `folium` or `ipyleaflet` using `shared_utils.map_utils`

References: 
* https://cityoflosangeles.github.io/best-practices/data-analysis-intermediate.html
* https://docs.calitp.org/data-infra/analytics_tools/python_libraries.html
* https://docs.calitp.org/data-infra/analytics_examples/warehouse_tutorial.html
* https://docs.calitp.org/data-infra/analytics_examples/new_tutorial.html

In [251]:
import geopandas as gpd
import intake
import pandas as pd

from calitp.tables import tbl
from calitp import query_sql
from siuba import *

from shapely.geometry import Polygon, LineString, Point
from shapely import wkt
from shapely.geometry import Point, Polygon
from shapely.wkt import loads

import shared_utils
import altair as alt
from shared_utils import altair_utils 

import branca
# Hint: if this doesn't import: refer to docs for correctly import
# cd into _shared_utils folder, run the make setup_env command

pd.options.display.float_format = "{:.2f}".format

## Research Question: What's the average number of trips per stop by operators in southern California? Show visualizations at the operator and county-level.
<br>**Geographic scope:** southern California counties
<br>**Deliverables:** chart(s) and map(s) showing metrics comparing across counties and also across operators. Make these visualizations using function(s).

### Prep data

* Use the same query, but grab a different set of operators. These are in southern California, so the map should zoom in counties ranging from LA to SD.
* To find what ITP IDs are what operators:
[agencies.yml](https://github.com/cal-itp/data-infra/blob/main/airflow/data/agencies.yml)
* *Hint*: for some counties, there are multiple operators. Make sure the average trips per stop by counties is the weighted average.
* Use the same [shapefile for CA counties](https://gis.data.ca.gov/datasets/CALFIRE-Forestry::california-county-boundaries/explore?location=37.246136%2C-119.002032%2C6.12) as in Exercise 4.
* Join the data and only keep counties that have bus stops.


In [252]:
#choosing a different set of ITP IDS
ITP_ID = [182, 183, 278, 154, 235, 232]

In [253]:
stops = (
    tbl.views.gtfs_schedule_dim_stops()
    >> select(_.itp_id == _.calitp_itp_id, _.stop_id, 
             _.stop_lat, _.stop_lon, _.stop_name)
    >> arrange(_.itp_id, _.stop_id, 
               _.stop_lat, _.stop_lon)
    >> collect()
    >> filter(_.itp_id.isin(ITP_ID))
    >> distinct()
)

In [254]:
#creating point geometry from longtitude &  latitude 
stops = shared_utils.geography_utils.create_point_geometry(stops, 'stop_lon', 'stop_lat')

In [255]:
stops.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487)
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069)


In [256]:
#checking CRS
stops.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [257]:
#bringing in CA dataframe
geojson = gpd.read_file('https://opendata.arcgis.com/datasets/8713ced9b78a4abb97dc130a691a8695_0.geojson').to_crs(epsg=4326)

In [258]:
#Joining the 2 dataframes, only keeping stops that are in a county
stops_joined = gpd.sjoin(stops, geojson, how='inner')

### Bring in a new table from BigQuery

* In `transitstacks`, there's a table called `ntd_stats`. 
* Decide what columns to keep.
* Merge `ntd_stats` with the `stops` table to create 1 df.

In [259]:
ntd_stats = (tbl.transitstacks.ntd_stats()
             >> select(_.transit_provider, _.itp_id, _.upt_total_2019)
             >> collect()
             >> filter(_.itp_id.isin(ITP_ID))
            )

In [260]:
#getting rid of commas & assigning to int to sum up later. 
ntd_stats = ntd_stats.assign(
    upt_total_2019 = ntd_stats.upt_total_2019.replace(',','', regex=True)).astype({"upt_total_2019": int}) 

### Merging the stops with ntd_stats - HELP
* Make sure all stop ids are unique
* When I dropped duplicates for stop ID to make sure I get only unique IDS, it dropped unique ones too? See above...

In [261]:
df2 = stops_joined.merge(ntd_stats, how = 'left', on ='itp_id')

In [262]:
df2[(df2['stop_id'].isin(riverside.stop_id)) & (df2.COUNTY_NAME == 'Riverside')].drop_duplicates(subset='stop_id')

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
31056,232,7593,33.98,-117.37,Riverside Metro Link Sb Lat,POINT (-117.37136 33.97539),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31057,232,8087,34.00,-117.05,County Line @ Fourth Wb Fs,POINT (-117.05295 34.00412),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31058,232,8603,33.98,-117.37,University @ Lemon Eb Ns,POINT (-117.37242 33.98087),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31059,232,8815,33.99,-117.55,Goodman @ Goodman Nb Fs 01,POINT (-117.55461 33.99433),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31060,235,8341,33.90,-117.47,LA SIERRA METROLINK STATION,POINT (-117.46784 33.90066),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,Orange County Transportation Authority,40743654


In [263]:
f'out of the {len(df2)} rows, only {df2.stop_id.nunique()} are unique'

'out of the 44589 rows, only 22356 are unique'

In [264]:
#new dataframe with only unique stop ids & trips per stop
df3 = df2.drop_duplicates(subset=['stop_id','itp_id'])

In [265]:
#checking that the rows make sense.
len(df3)

29538

In [266]:
#filtered for unique stops ids 
df3[(df3.COUNTY_NAME.str.contains("Riverside", case= False))]

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
31056,232,7593,33.98,-117.37,Riverside Metro Link Sb Lat,POINT (-117.37136 33.97539),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31057,232,8087,34.00,-117.05,County Line @ Fourth Wb Fs,POINT (-117.05295 34.00412),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31058,232,8603,33.98,-117.37,University @ Lemon Eb Ns,POINT (-117.37242 33.98087),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31059,232,8815,33.99,-117.55,Goodman @ Goodman Nb Fs 01,POINT (-117.55461 33.99433),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,OmniTrans,10863530
31060,235,8341,33.90,-117.47,LA SIERRA METROLINK STATION,POINT (-117.46784 33.90066),32,33,Riverside,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,Orange County Transportation Authority,40743654


In [267]:
#not filtered, five stop ids that are all unique?
riverside = stops_joined[(stops_joined.COUNTY_NAME.str.contains("Riverside", case= False))]

### Aggregate 
<b> Instructions </b> 
* Write a function to aggregate to the operator level or county level, add new columns for desired metrics.
* Merge in CA shapefile to get a gdf.
* Add another `geometry` column, called `centroid`, and grab the county's centroid.
* Refer to [docs](https://geopandas.org/en/stable/docs/reference/api/geopandas.GeoDataFrame.set_geometry.html) to see how to pick which column to use as the `geometry` for the gdf, since technically, a gdf can handle multiple geometry columns.



In [268]:
df3.head(2)

,itp_id,stop_id,stop_lat,stop_lon,stop_name,geometry,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,transit_provider,upt_total_2019
0,154,19113,33.54,-117.78,Laguna Beach Bus Depot,POINT (-117.78271 33.54487),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829
1,154,19114,33.54,-117.78,Legion Hall,POINT (-117.78006 33.54069),29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,Laguna Beach Municipal Transit,820829


In [269]:
#County Subsets
county_stops = df3[['geometry','stop_id','COUNTY_NAME']]
county_passengers = df3[['geometry','upt_total_2019','COUNTY_NAME']]

In [270]:
#transit providers
transit_stops = df3[['geometry','stop_id','transit_provider']]
transit_passengers = df3[['geometry','upt_total_2019','transit_provider']]

In [271]:
#function 2
def aggregate2 (df1, df2, group_by_col):
    df1 = df1.dissolve(by = group_by_col, aggfunc= 'nunique').rename(columns = {'stop_id':'total_stops'})
    df2 = df2.dissolve(by = group_by_col, aggfunc= 'sum').rename(columns = {'upt_total_2019':'total_trips'})
    df3 = df1.merge(df2, how = 'inner', on = [group_by_col, 'geometry'])
    #find sum up total trips 
    df3['total_trips_sum'] = df3['total_trips'].sum() 
    #multiply total stops by total trips within a transit operator / county  
    df3['frequency'] = df3['total_stops']*df3['total_trips']
    #divide
    df3['weighted_avg'] = df3['frequency']/df3['total_trips_sum'] 
    #drop old cols
    #df3 = df3.drop(columns=['frequency','total_stops_sum'])
    df3 = df3.reset_index()
    return df3

In [272]:
def aggregate3(df, group_by_col):
    df1 = df.groupby(group_by_col).agg({"stop_id": "nunique", 
                                           "upt_total_2019": "sum"}).reset_index()
    df = df[['geometry',group_by_col]]
    df2 = pd.merge(df.drop_duplicates(), 
                   df1, 
                   on = group_by_col, 
                   how = "left", 
                   validate = "1:1",
                  )
    
    return df2

In [273]:
county_agg = aggregate2(county_stops, county_passengers, 'COUNTY_NAME')
county_agg

,COUNTY_NAME,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg
0,Los Angeles,"MULTIPOINT (-118.84339 34.03152, -118.82790 34...",16900,5314912123103,5980688001300,89822014880440700,15018.68
1,Orange,"MULTIPOINT (-118.10607 33.74889, -118.10577 33...",5581,239217387671,5980688001300,1335072240591851,223.23
2,Riverside,"MULTIPOINT (-117.55461 33.99433, -117.46784 33...",5,84197774,5980688001300,420988870,0.00
3,San Bernardino,"MULTIPOINT (-117.73260 34.00136, -117.73217 33...",2331,25322888430,5980688001300,59027652930330,9.87
4,San Diego,"MULTIPOINT (-117.27792 32.83915, -117.27780 32...",4569,389998394655,5980688001300,1781902665178695,297.94
5,Ventura,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",55,11153009667,5980688001300,613415531685,0.10


In [274]:
transit_agg= aggregate2(transit_stops, transit_passengers, 'transit_provider')
transit_agg

,transit_provider,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg
0,Laguna Beach Municipal Transit,"MULTIPOINT (-117.80188 33.54913, -117.79987 33...",94,77157926,5980688001300,7252845044,0.00
1,Los Angeles Department of Transportation,"MULTIPOINT (-118.88283 34.18006, -118.88280 34...",3033,58514689341,5980688001300,177475052771253,29.67
2,Los Angeles Metro,"MULTIPOINT (-118.86092 34.17506, -118.85796 34...",13902,5278841318142,5980688001300,73386452004810084,12270.57
3,OmniTrans,"MULTIPOINT (-117.75124 34.05917, -117.75041 34...",2351,25540159030,5980688001300,60044913879530,10.04
4,Orange County Transportation Authority,"MULTIPOINT (-118.28095 33.96019, -118.28079 33...",5589,227716282206,5980688001300,1272706301249334,212.80
5,San Diego Metropolitan Transit System,"MULTIPOINT (-117.27792 32.83915, -117.27780 32...",4569,389998394655,5980688001300,1781902665178695,297.94


#### Centroids
* For transit_agg: "/tmp/ipykernel_1946/1881472231.py:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation." 

In [275]:
county_agg2 = pd.merge(county_agg, geojson,  on=['COUNTY_NAME']).drop(columns = ['geometry_x']).rename(columns = {'COUNTY_NAME':'County', 'geometry_y':'geometry'}) 
county_agg2 = county_agg2.loc[county_agg2['ISLAND'] != 'Channel Islands'] 
county_agg2['centroid'] = county_agg2.geometry.centroid
county_agg2

/tmp/ipykernel_1074/2017153488.py:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.



,County,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry,centroid
0,Los Angeles,16900,5314912123103,5980688001300,89822014880440700,15018.68,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672...",POINT (-118.21689 34.36117)
3,Orange,5581,239217387671,5980688001300,1335072240591851,223.23,30,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150...",POINT (-117.76121 33.70309)
4,Riverside,5,84197774,5980688001300,420988870,0.00,33,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356...",POINT (-115.99382 33.74374)
5,San Bernardino,2331,25322888430,5980688001300,59027652930330,9.87,36,SBD,36,36,071,None,{E4DF6870-0D0B-40C1-AD3B-A60A72792CFD},10.46,5.13,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970...",POINT (-116.17852 34.84146)
6,San Diego,4569,389998394655,5980688001300,1781902665178695,297.94,37,SDG,37,37,073,None,{414826EC-689E-4084-BD03-5195DF2748BF},4.73,1.06,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413...",POINT (-116.73509 33.03422)
7,Ventura,55,11153009667,5980688001300,613415531685,0.10,56,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,"MULTIPOLYGON (((-119.35649 34.87366, -119.3564...",POINT (-119.07833 34.47124)


### Visualizations
* Make one chart for comparing trips per stop by operators, and another chart for comparing it by counties. Use a function to do this.
* Make 1 map for comparing trips per stop by counties. Use `shared_utils.map_utils` to do this.
* Visualizations should follow the Cal-ITP style guide.
* More on `folium` and `ipyleaflet`: https://github.com/jorisvandenbossche/geopandas-tutorial/blob/master/05-more-on-visualization.ipynb

#### Scatter Chart Function

In [276]:
def scatter_chart(df, x_col, y_col):
    y_col_stripped = y_col.replace('_',' ')
    x_col_stripped = x_col.replace('_',' ')
    chart_title = f"{y_col_stripped} by {x_col_stripped}" 
    chart = (alt.Chart(df).mark_circle(size=500).encode(
                 x=alt.X(x_col, title=x_col_stripped),
                 y=alt.Y(y_col, title=y_col_stripped),   
                 color =alt.Color(x_col, scale=alt.Scale(range=altair_utils.CALITP_CATEGORY_BOLD_COLORS)),
                 tooltip = [alt.Tooltip(x_col),alt.Tooltip(y_col)]).interactive().properties(width=600,height=250,
                 title = chart_title)
            )
    #return chart
    display(chart)

In [277]:
transit_agg2.head(1)

,transit_provider,geometry,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,index_right,OBJECTID,COUNTY_NAME,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,centroid
0,Laguna Beach Municipal Transit,"MULTIPOINT (-117.80188 33.54913, -117.79987 33...",94,77157926,5980688001300,7252845044,0.00,29,30,Orange,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,POINT (-117.76699 33.53023)


In [278]:
county_agg2.head(1)

,County,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry,centroid
0,Los Angeles,16900,5314912123103,5980688001300,89822014880440700,15018.68,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672...",POINT (-118.21689 34.36117)


### Function for a bunch of charts

In [279]:
def charts(df, x_axis_col):
    for i in ['total_stops', 'total_trips','weighted_avg']:
        df_i = df[[x_axis_col, i]].drop_duplicates()
        scatter_chart_i = scatter_chart(df_i, x_axis_col, i)
        display (scatter_chart_i)
    return scatter_chart_i

In [280]:
charts(transit_agg2, 'transit_provider')

alt.Chart(...)

None

alt.Chart(...)

None

alt.Chart(...)

None

In [281]:
charts(county_agg2, 'County')

alt.Chart(...)

None

alt.Chart(...)

None

alt.Chart(...)

None

#### Map

In [282]:
choropleth_dict = ({
            "layer_name": 'my layer',
            "MIN_VALUE": int(0),
            "MAX_VALUE": int(100),
            "plot_col_name": 'County_Name',
            "fig_width": '100%',
            "fig_height": '100%',
            "fig_min_width_px": '600px',
            "fig_min_height_px": '600px'})

In [283]:
test = county_agg2.drop(columns = ['centroid']).rename(columns = {'geometry_y':'geometry'})
test

,County,total_stops,total_trips,total_trips_sum,frequency,weighted_avg,OBJECTID,COUNTY_ABBREV,COUNTY_NUM,COUNTY_CODE,COUNTY_FIPS,ISLAND,GlobalID,SHAPE_Length,SHAPE_Area,geometry
0,Los Angeles,16900,5314912123103,5980688001300,89822014880440700,15018.68,19,LOS,19,19,037,None,{3B1E1D69-2B1A-464D-BA43-611C4201B219},5.18,1.00,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672..."
3,Orange,5581,239217387671,5980688001300,1335072240591851,223.23,30,ORA,30,30,059,None,{E9A24C4D-A4EC-40B5-B689-89DE5354A669},2.16,0.20,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150..."
4,Riverside,5,84197774,5980688001300,420988870,0.00,33,RIV,33,33,065,None,{5A8DE123-B918-4725-9868-268A4BBCC4E7},7.95,1.84,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356..."
5,San Bernardino,2331,25322888430,5980688001300,59027652930330,9.87,36,SBD,36,36,071,None,{E4DF6870-0D0B-40C1-AD3B-A60A72792CFD},10.46,5.13,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970..."
6,San Diego,4569,389998394655,5980688001300,1781902665178695,297.94,37,SDG,37,37,073,None,{414826EC-689E-4084-BD03-5195DF2748BF},4.73,1.06,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413..."
7,Ventura,55,11153009667,5980688001300,613415531685,0.10,56,VEN,56,56,111,None,{DFDE8E6B-742D-48C4-A4A0-4952E37D938B},3.00,0.47,"MULTIPOLYGON (((-119.35649 34.87366, -119.3564..."


In [284]:
colorscale = branca.colormap.StepColormap(
                colors=["gray", "green", "navy"], 
                #index=[2_000, 4_000, 8_000],
                #vmin=0, vmax=15_000,
)


In [297]:
REGION_CENTROIDS = shared_utils.map_utils.REGION_CENTROIDS

In [299]:
REGION_CENTROIDS??

Type:        dict
String form: {'Alameda': [[37.84, -122.27], 12], 'Los Angeles': [[34.0, -118.18], 11], 'CA': [[35.8, -119.4], 6]}
Length:      3
Docstring:  
dict() -> new empty dictionary
dict(mapping) -> new dictionary initialized from a mapping object's
    (key, value) pairs
dict(iterable) -> new dictionary initialized as if via:
    d = {}
    for k, v in iterable:
        d[k] = v
dict(**kwargs) -> new dictionary initialized with the name=value pairs
    in the keyword argument list.  For example:  dict(one=1, two=2)


In [287]:
display_cols = ["weighted_avg", "County", "geometry"]
test[display_cols]

,weighted_avg,County,geometry
0,15018.68,Los Angeles,"MULTIPOLYGON (((-117.66733 34.79317, -117.6672..."
3,223.23,Orange,"MULTIPOLYGON (((-118.11526 33.74164, -118.1150..."
4,0.00,Riverside,"MULTIPOLYGON (((-114.43559 34.07847, -114.4356..."
5,9.87,San Bernardino,"MULTIPOLYGON (((-115.41359 35.62499, -115.3970..."
6,297.94,San Diego,"MULTIPOLYGON (((-117.24152 33.44879, -117.2413..."
7,0.10,Ventura,"MULTIPOLYGON (((-119.35649 34.87366, -119.3564..."


In [288]:
test.geometry.name

'geometry'

In [289]:
colorscale

In [290]:
#test = test.set_geometry("centroid")

In [291]:
REGION_CENTROIDS["Alameda"]

[[37.84, -122.27], 12]

In [294]:
REGION_CENTROIDS

{'Alameda': [[37.84, -122.27], 12],
 'Los Angeles': [[34.0, -118.18], 11],
 'CA': [[35.8, -119.4], 6]}

In [292]:
shared_utils.map_utils.make_ipyleaflet_choropleth_map??

Signature:
shared_utils.map_utils.make_ipyleaflet_choropleth_map(
    gdf,
    plot_col,
    geometry_col,
    choropleth_dict,
    colorscale,
    zoom=6,
    centroid=[35.8, -119.4],
    **kwargs,
)
Source:   
def make_ipyleaflet_choropleth_map(gdf, plot_col, geometry_col,
                                   choropleth_dict,
                                   colorscale, 
                                   zoom=REGION_CENTROIDS["CA"][1], 
                                   centroid = REGION_CENTROIDS["CA"][0], 
                                   **kwargs,
                                  ):
    '''
    Parameters:
    
    gdf: geopandas.GeoDataFrame
    plot_col: str, name of the column to map
    geometry_col: str, e.g., "Tract" or "District"
    
    choropleth_dict: dict. Takes this format:
        {
            "layer_name": string, human-readable layer name,
            "MIN_VALUE": numeric,
            "MAX_VALUE": numeric,
            "plot_col_name": string, human-readable c

In [296]:
shared_utils.map_utils.make_ipyleaflet_choropleth_map(test, 
                                                     plot_col = 'weighted_avg',
                                                     geometry_col = 'County', 
                                                     choropleth_dict = choropleth_dict, 
                                                     colorscale = colorscale,
                                                     zoom=9, 
                                                     centroid = REGION_CENTROIDS[Los Angeles]["centroid"])

SyntaxError: invalid syntax (3994864892.py, line 7)